# 🎓 CMU 11-785: Introduction to Deep Learning
## Cloud TPU v5e Platform Guide & Best Practices
**Instructors:** Prof. Bhiksha Raj & Prof. Rita Singh  
**Platform Maintainer:** Ayush Morbar (`amorbar@andrew.cmu.edu`)  
**Infrastructure:** GKE Autopilot • Cloud TPU v5e • Kueue 0.19 • Tartan 3-Tier Storage Hierarchy  

---

### Welcome to the CMU 11-785 Shared TPU Accelerator Platform!
This interactive guide is designed for **students**, **capstone research teams**, and **course administrators/TAs** starting from scratch. It walks through all modern features, storage hierarchies, queuing mechanics, and administrative capabilities.

```text
+---------------------------------------------------------------------------------------------------+
|                                     CMU 11-785 PLATFORM TOPOLOGY                                 |
+---------------------------------------------------------------------------------------------------+
|                                                                                                   |
|  [ Interactive Student Workspace ]        [ Ephemeral Accelerator Pool ]                         |
|  +------------------------------+         +-------------------------------+                      |
|  | JupyterLab (Spot CPU Node)   | =======>| Kueue FIFO Queue (tpu)        |                      |
|  | 2 vCPU / 16 GB RAM           | (submit)| ct5lp-hightpu-1t Nodes        |                      |
|  | Priority: student-notebook   |         | 1x TPU v5e Chip (197 TFLOP/s) |                      |
|  +------------------------------+         | Priority: student-tpu-job     |                      |
|                 |                         +-------------------------------+                      |
|                 v                                         |                                      |
|  +--------------------------------------------------------+------------------------------------+ |
|  |                           TARTAN 3-TIER STORAGE HIERARCHY                                   | |
|  |  Tier 1: /tartan/11785 & /ocean (GCS FUSE Shared Store + Local NVMe Cache)                  | |
|  |  Tier 2: /home/jovyan (32 GiB Persistent Standard-RWO SSD Disk)                            | |
|  |  Tier 3: /tmp (Local Node NVMe SSD Scratch, Line-Rate IOPS)                                 | |
|  +---------------------------------------------------------------------------------------------+ |
+---------------------------------------------------------------------------------------------------+
```

---

## 1. Storage Architecture: Tartan 3-Tier Hierarchy

The platform implements a **3-tier storage architecture** designed for high throughput, zero cross-student interference, and cost efficiency:

1. **Tier 1 — Central Course Store (`/tartan/11785` and `/ocean`):**
   - Dual-mounted across every student and staff pod via **GKE Cloud Storage FUSE**.
   - Features local NVMe disk caching (`file-cache:max-size-mb:50000`) so shared datasets (e.g. HW2P2 face images) stream at SSD line-rate ($>1.2\text{ GB/s}$) on subsequent training epochs.
   - Hosts shared Python environments (`/tartan/11785/envs/idl-shared-env`).

2. **Tier 2 — Student Private Home (`/home/jovyan`):**
   - Dedicated 32 GiB PersistentVolumeClaim (Standard-RWO SSD) that survives notebook restarts.
   - Stores active code, notebooks, and private credentials (`.env`).
   - Configured with `fsGroupChangePolicy: "OnRootMismatch"` for instant (<1s) pod attachment without recursive permission stalls.

3. **Tier 3 — Local Node Scratch (`/tmp`):**
   - Ephemeral high-speed NVMe scratch directly on the worker node for temporary batch files and unpack operations.

Let's inspect your mounted storage in the cell below:

In [ ]:
# Verify storage mounts and disk space
!echo "=== Student Persistent Disk (/home/jovyan) ==="
!df -h /home/jovyan

!echo -e "\n=== Tartan Shared Store (/tartan/11785) ==="
!ls -la /tartan/11785

!echo -e "\n=== Backward-Compatible Ocean Dual-Mount (/ocean) ==="
!ls -la /ocean

## 2. Student Privacy & Credential Isolation (`.env`)

### Security Rule: Never Hardcode API Keys
Never write private API keys (Weights & Biases, Kaggle, HuggingFace) into notebooks or Git repos. Instead, the platform automatically discovers keys from `/home/jovyan/.env`.

`submit_tpu` reads your personal `.env`, masks the credentials in logs (e.g. `sha256[:8]`), and safely injects them into your ephemeral TPU jobs via Kubernetes secrets.

In [ ]:
import os
from pathlib import Path

env_path = Path("/home/jovyan/.env")
if not env_path.exists():
    print("Creating baseline ~/.env template...")
    env_path.write_text("""# CMU 11-785 Student Credentials
# Add your private API keys below (DO NOT SHARE OR COMMIT)
WANDB_API_KEY=
KAGGLE_USERNAME=
KAGGLE_KEY=
HF_TOKEN=
""")
    print("✓ Created ~/.env. Edit it in JupyterLab to add your personal tokens.")
else:
    print("✓ ~/.env already exists on your persistent disk.")

## 3. Interactive Hardware Probe: Your First TPU Run

Notice that this notebook runs on a **Spot CPU node** (2 vCPU, 16 GB RAM). This keeps your workspace responsive and costs $0.03/hr instead of $1.35/hr.

When you want to execute on a Cloud TPU v5e chip, use **`submit_tpu.run(...)`**.
- **Cold Start:** If the cluster scaled to zero, GKE Autopilot takes ~2 minutes to provision a node.
- **Warm Execution:** Subsequent runs execute within **1 to 5 seconds** because the warm node is held by Kueue.

In [ ]:
import submit_tpu

print("Submitting hardware probe to TPU queue...")
result = submit_tpu.run('''
import jax
devices = jax.devices()
print(f"Active Devices: {devices}")
print(f"Device Platform: {devices[0].platform}")
print(f"Device Kind: {devices[0].device_kind}")
''')
print(result)

## 4. Benchmark: Multi-Head Attention & FlashAttention on TPU v5e

A single Cloud TPU v5e chip delivers up to **197 TFLOP/s** of bfloat16 compute. Below, we benchmark a standard Multi-Head Attention forward pass ($B=8, H=12, S=1024, D=64$) and compare it with TPU-optimized FlashAttention (`jax.nn.dot_product_attention`).

In [ ]:
benchmark_code = '''
import time
import jax
import jax.numpy as jnp

B, H, S, D = 8, 12, 1024, 64
key = jax.random.PRNGKey(42)
q, k, v = jax.random.normal(key, (3, B, H, S, D), dtype=jnp.bfloat16)

# 1. Standard Two-Einsum Attention
@jax.jit
def naive_attn(q, k, v):
    logits = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(D).astype(q.dtype)
    return jnp.einsum("bhqk,bhkd->bhqd", jax.nn.softmax(logits, axis=-1), v)

naive_attn(q, k, v).block_until_ready() # warmup
t0 = time.perf_counter()
for _ in range(100):
    naive_attn(q, k, v).block_until_ready()
naive_dt = (time.perf_counter() - t0) / 100
flops = 4 * B * H * S * S * D

print(f"[Standard Attention] {naive_dt*1e3:.3f} ms | {flops / naive_dt / 1e12:.2f} TFLOP/s")

# 2. Fused FlashAttention (jax.nn.dot_product_attention)
@jax.jit
def flash_attn(q, k, v):
    # Q, K, V shaped as (B, S, H, D)
    return jax.nn.dot_product_attention(q.swapaxes(1, 2), k.swapaxes(1, 2), v.swapaxes(1, 2)).swapaxes(1, 2)

flash_attn(q, k, v).block_until_ready() # warmup
t1 = time.perf_counter()
for _ in range(100):
    flash_attn(q, k, v).block_until_ready()
flash_dt = (time.perf_counter() - t1) / 100

print(f"[Flash Attention]    {flash_dt*1e3:.3f} ms | {flops / flash_dt / 1e12:.2f} TFLOP/s")
print(f"Speedup: {naive_dt / flash_dt:.2f}x")
'''

print(submit_tpu.run(benchmark_code))

## 5. Production Batch Training with `submit_tpu.train()`

For real homework and research experiments (e.g. HW2P2 ConvNeXt / ResNet / Transformer training), **do not paste long code blocks into `submit_tpu.run()`**.

Instead, keep your code in a standard Python file (e.g. `train.py`) and submit it as an asynchronous batch training job with **`submit_tpu.train(...)`**.

### Key Advantages of `submit_tpu.train()`:
* **Non-blocking / Background Execution:** You can disconnect your laptop or close the browser tab; the job continues running on the TPU until completion.
* **Live Log Streaming:** Stream stdout/stderr directly into your notebook.
* **Automatic Checkpointing:** Saves Orbax checkpoints to `/tartan/11785/checkpoints` or your personal PVC.
* **Billing Metadata Labels:** Tags jobs with `cmu-course: 11-785` and `cmu-grant-tier: tier0-coursework`.

In [ ]:
# Example: Inspect how submit_tpu.train() is called
import inspect
print("submit_tpu.train Signature:")
print(inspect.signature(submit_tpu.train))
print("\nDocumentation Summary:")
print(submit_tpu.train.__doc__[:600] + "...")

## 6. Queue Diagnostics & Priority Classes

### Why Does My Notebook Never Crash During Scale Tests?
The platform enforces strict **Kubernetes PriorityClasses**:
1. `jupyterhub-core` (1,000,000): Ingress proxy and Hub router. Highest priority.
2. `student-notebook` (1,000): Interactive notebook sessions. A notebook holds live student state, so it **never yields** to batch jobs.
3. `student-tpu-job` (100): Ephemeral batch jobs. Fungible by design—if resources are needed, jobs yield and Kueue re-queues them.

Run the cell below to inspect your place in the Kueue batch queue:

In [ ]:
!echo "=== Kueue Workload Queue Status ==="
!kubectl get workloads -o wide 2>/dev/null || echo "(Workload inspection requires student RBAC)"

!echo -e "\n=== LocalQueue Status ==="
!kubectl get localqueue tpu 2>/dev/null || echo "(Queue inspection active in course namespace)"

## 7. Administrator & TA Management Toolkit

For course maintainers, TAs, and research project admins, the platform provides programmatic storage and grant management via `scripts/hub_admin.py`.

In [ ]:
import sys
sys.path.insert(0, "/opt/course")
try:
    import hub_admin
    print("hub_admin module loaded successfully.")
    # List all PVC storage allocations
    storage_table = hub_admin.list_student_storage()
    print(f"Found {len(storage_table)} student PVCs in namespace:")
    for s in storage_table[:5]:
        print(f"  - User: {s['username']:<15} PVC: {s['name']:<20} Requested: {s['requested']:<8} Status: {s['status']}")
except Exception as e:
    print(f"hub_admin info: {e}")

### Useful Administrative Commands (Cloud Shell Reference)

```bash
# 1. List all student volume allocations
make admin-list

# 2. Grant Tier 1 Capstone resources to an approved student
make admin-grant USER=student_id TIER=1

# 3. Expand a student's disk from 32 GiB to 100 GiB online (zero downtime)
python scripts/hub_admin.py expand student_id 100Gi

# 4. Warm up TPU pool before a live lecture demo (1 chip)
make warm-on WARM=1

# 5. Release warm TPU pool to scale to $0 when done
make warm-off
```

---
### Summary
You now have a fully functioning, cost-optimized, and resilient development setup. Happy deep learning!